# METHOD 1 — Similarity & Rules
### K-Nearest Neighbors versus Decision Trees

**Big question:** How can a computer recognize a doodle without understanding what it depicts?

This notebook is fully implemented. Run cells top to bottom. Experiment by changing k and tree depth; no coding from scratch required.

**Presenter:** 8 minutes. Show nearest neighbors and a tiny decision tree. **Never use the test split for tuning.**

In [ ]:
# Shared setup: works from local Jupyter or a fresh Google Colab runtime.
import subprocess, sys
from pathlib import Path
here = Path.cwd().resolve()
candidates = [here, here.parent, here / "doodle-ml-workshop", here.parent / "doodle-ml-workshop"]
ROOT = next((p for p in candidates if (p / "src" / "data.py").exists()), None)
if ROOT is None:
    checkout = Path("/content/Computer-Vision-Workshop")
    if not checkout.exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", "doodle-ai-workshop",
            "https://github.com/jaekop/Computer-Vision-Workshop.git", str(checkout)
        ], check=True)
    ROOT = checkout / "doodle-ml-workshop"
sys.path.insert(0, str(ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])
print("Workshop ready at", ROOT)

## 1. Our labeled doodles
Google Quick, Draw! 28×28 grayscale bitmaps are flattened into 784 pixel values. A label tells us the category. The data loader downloads only a small prefix of each official class file and creates fixed, reproducible splits.

In [ ]:
from src.data import load_data
from src.evaluation import show_examples, validation_accuracy, show_mistakes
from src.models import fit_model
import matplotlib.pyplot as plt
import numpy as np

data = load_data(cache_dir=ROOT / "data")
print("Categories:", ", ".join(data.class_names))
print("Training:", data.X_train.shape, "Validation:", data.X_val.shape)
print("Test is held back until the Final Challenge.")
show_examples(data.X_train, data.y_train, data.class_names, count=12)
plt.show()

## 2. KNN — ask the neighbors
For a new drawing, KNN measures distances to training examples. The nearest k examples vote. A pixel distance is not the same thing as human visual similarity.

In [ ]:
knn_model = fit_model(data.X_train, data.y_train, "knn", representation="pixels", n_neighbors=5)
print(f"KNN validation accuracy: {validation_accuracy(knn_model, data.X_val, data.y_val):.3f}")
query = data.X_val[:1]
distances, neighbors = knn_model.estimator.kneighbors(knn_model.features(query), n_neighbors=5)
fig, axes = plt.subplots(1, 6, figsize=(13, 3))
axes[0].imshow(query[0].reshape(28, 28), cmap="gray")
axes[0].set_title("New drawing")
for ax, idx, distance in zip(axes[1:], neighbors[0], distances[0]):
    ax.imshow(data.X_train[idx].reshape(28, 28), cmap="gray")
    ax.set_title(f"{data.class_names[int(data.y_train[idx])]}\nd={distance:.1f}", fontsize=9)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

### Try it: change k
Prediction: will k=1 be more sensitive to odd training examples than k=11? Check the validation set.

In [ ]:
ks = [1, 3, 5, 11, 21]
knn_scores = []
for k in ks:
    m = fit_model(data.X_train, data.y_train, "knn", n_neighbors=k)
    knn_scores.append(validation_accuracy(m, data.X_val, data.y_val))
plt.plot(ks, knn_scores, marker="o", color="#6F53D8")
plt.xlabel("Number of neighbors (k)")
plt.ylabel("Validation accuracy")
plt.title("KNN: changing k changes model behavior")
plt.ylim(0, 1); plt.grid(alpha=.2); plt.show()
print(dict(zip(ks, [round(v, 3) for v in knn_scores])))

## 3. Decision Tree — learn yes/no questions
Trees split feature values at learned thresholds. Deeper trees can memorize quirks of the training set.

In [ ]:
from sklearn.tree import plot_tree
tree_model = fit_model(data.X_train, data.y_train, "tree", max_depth=12)
print(f"Tree validation accuracy: {validation_accuracy(tree_model, data.X_val, data.y_val):.3f}")
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(tree_model.estimator, max_depth=2, filled=True, rounded=True,
          class_names=list(data.class_names), fontsize=8, ax=ax)
ax.set_title("Only the first three levels of a much larger learned tree")
plt.show()

### Try it: change tree depth
What happens to training accuracy and validation accuracy as the tree gets deeper?

In [ ]:
from sklearn.metrics import accuracy_score
depths = [2, 4, 8, 12, 20, None]
train_scores, val_scores = [], []
for depth in depths:
    m = fit_model(data.X_train, data.y_train, "tree", max_depth=depth)
    train_scores.append(accuracy_score(data.y_train, m.predict(data.X_train)))
    val_scores.append(validation_accuracy(m, data.X_val, data.y_val))
labels = [str(d) for d in depths]
plt.plot(labels, train_scores, "o-", label="Training")
plt.plot(labels, val_scores, "o-", label="Validation")
plt.xlabel("Maximum depth"); plt.ylabel("Accuracy"); plt.ylim(0, 1)
plt.legend(); plt.title("More complexity is not always better"); plt.show()

## 4. Same doodle, two inductive biases
KNN asks **which examples are close?** A tree asks **which thresholds are crossed?** Neither must win on every dataset.

In [ ]:
i = 7
plt.imshow(data.X_val[i].reshape(28, 28), cmap="gray")
plt.axis("off"); plt.show()
print("True label:", data.class_names[int(data.y_val[i])])
print("KNN:", data.class_names[int(knn_model.predict(data.X_val[i:i+1])[0])])
print("Tree:", data.class_names[int(tree_model.predict(data.X_val[i:i+1])[0])])
show_mistakes(tree_model, data.X_val, data.y_val, count=8)
plt.show()

### Copy this into the Final Challenge
- Use `fit_model(..., "knn", n_neighbors=5)` for similarity.
- Use `fit_model(..., "tree", max_depth=12)` for learned rules.
- Tune with **validation**, not test. Next: can many trees vote together?